# Pairs Trading — Exploration Notebook

End-to-end walkthrough of the pipeline in `src/`:

1. **Cointegration screening** — OLS hedge ratio + ADF stationarity test on a candidate pair.
2. **Signal generation** — rolling z-score of the spread, entry/exit rules.
3. **Backtest** — event-driven P&L with slippage, plus a walk-forward variant that re-fits the hedge ratio out-of-sample.
4. **Performance metrics** — Sharpe, drawdown, win rate, payoff ratio.

Example pair: `RIO.L` / `BHP.L` (two London-listed mining majors), split into a training
window (fit the hedge ratio) and a held-out test window (evaluate signals/backtest).


In [ ]:
import sys
from pathlib import Path

# Make `src/` importable when running this notebook from notebooks/.
sys.path.insert(0, str(Path.cwd().parent))

from src.cointegration import load_prices, fit_ols, calc_spread, adf_test, screen_pair, scan_pairs
from src.signals import calc_zscore, generate_signals
from src.backtest import run_backtest, walk_forward_backtest
from src.metrics import perf_metrics, print_metrics
from src.plotting import plot_adf_spread, plot_spread_zscore, plot_backtest

TICKER_1 = "RIO.L"
TICKER_2 = "BHP.L"
WINDOW   = 60
ENTRY_Z  = 2.0
EXIT_Z   = 0.0

TRAIN_START, TRAIN_END = "2018-01-01", "2020-12-31"
TEST_START,  TEST_END  = "2021-01-01", "2023-12-31"


## 1. Cointegration screening (training window)

In [ ]:
train_y, train_x = load_prices(TICKER_1, TICKER_2, TRAIN_START, TRAIN_END)

alpha, beta, ols_model = fit_ols(train_y, train_x)
train_spread = calc_spread(train_y, train_x, alpha, beta)
train_adf = adf_test(train_spread)

print(f"Hedge ratio (beta):  {beta:.4f}")
print(f"Intercept (alpha):   {alpha:.4f}")
print(f"ADF statistic:       {train_adf['adf_stat']:.4f}")
print(f"ADF p-value:         {train_adf['p_value']:.4f}")

plot_adf_spread(train_spread, train_adf, TICKER_1, TICKER_2, TRAIN_START, TRAIN_END, save_path=None)


### Optional: screen a wider universe

`scan_pairs` batches the download and ranks every combination by ADF statistic —
useful for finding candidate pairs before committing to one.


In [ ]:
candidates = ["RIO.L", "BHP.L", "AAL.L", "GLEN.L", "ANTO.L"]
rankings = scan_pairs(candidates, TRAIN_START, TRAIN_END, min_rows=200)
rankings.head(10)


## 2. Signal generation (test window)

In [ ]:
test_y, test_x = load_prices(TICKER_1, TICKER_2, TEST_START, TEST_END)

# Hedge ratio (alpha, beta) is carried over from the training fit — the test
# window is genuinely out-of-sample.
test_spread = calc_spread(test_y, test_x, alpha, beta)
test_zscore = calc_zscore(test_spread, WINDOW)
test_position = generate_signals(test_zscore, ENTRY_Z, EXIT_Z)

plot_spread_zscore(test_spread, test_zscore, save_path=None)


## 3. Backtest

In [ ]:
pnl_df, trade_log = run_backtest(test_y, test_x, alpha, beta, WINDOW, ENTRY_Z, EXIT_Z)

plot_backtest(pnl_df, save_path=None)

print("\n--- Trade Log ---")
print(trade_log.to_string())
print(f"\nTotal P&L (pre-cost): {pnl_df['CumulativePnL'].iloc[-1]:.2f}")


### Walk-forward variant

Re-fits alpha/beta on a rolling training window instead of a single fixed
in-sample fit, then trades the following block strictly out-of-sample before
sliding forward. More robust to the hedge ratio drifting over time.


In [ ]:
wf_pnl_df, wf_trade_log = walk_forward_backtest(
    TICKER_1, TICKER_2,
    start=TRAIN_START, end=TEST_END,
    train_window=252, test_window=63, zscore_window=WINDOW,
    entry_z=ENTRY_Z, exit_z=EXIT_Z,
)

if not wf_pnl_df.empty:
    plot_backtest(wf_pnl_df, save_path=None)
    print(f"Walk-forward total P&L (pre-cost): {wf_pnl_df['CumulativePnL'].iloc[-1]:.2f}")
else:
    print("Not enough history for a full train+test walk-forward block.")


## 4. Performance metrics

In [ ]:
fixed_hedge_metrics = perf_metrics(pnl_df, trade_log, cost=True)
print("--- Fixed hedge-ratio backtest ---")
print_metrics(fixed_hedge_metrics)

if not wf_pnl_df.empty:
    walk_forward_metrics = perf_metrics(wf_pnl_df, wf_trade_log, cost=True)
    print("\n--- Walk-forward backtest ---")
    print_metrics(walk_forward_metrics)
